# Week 2, Lab 3: Embeddings Deep Dive - Teaching Your Twin to Understand Meaning

## The map before the territory

Searching for the exact words "cook" or "bake" misses a sentence that says "she makes bread". A computer needs a way to compare **meaning**, not spelling. The trick is an **embedding**: a list of numbers that stands for the meaning of a word or sentence. Texts that mean similar things get lists of numbers that are close together.

Real embeddings come from a trained model and have hundreds or thousands of numbers. This lab uses tiny hand-made ones, with just 5 numbers per word, so that you can see every step. The ideas are exactly the same. You will write the one function that every embedding search needs: a score for "how close are these two lists of numbers?". There is no API call. The Reading tab explains lists of numbers, `zip`, `sum` and `math.sqrt` from scratch if you want a refresher first.

## Exercise

### Goal
Write `cosine_similarity`, a score between -1 and 1 for how much two vectors point the same way. A *vector* is just a list of numbers, like `[0.9, 0.1, 0.0]`. Think of it as an arrow. *Cosine similarity* ignores how long the arrows are and only looks at their direction. A score of `1.0` means the same direction (very similar meaning). `0.0` means a right angle (nothing in common). `-1.0` means opposite directions.

### 1. `cosine_similarity(a, b)`
Code shape (replace each `...`):
```python
def cosine_similarity(a, b):
    dot = sum(... for x, y in zip(a, b))
    length_a = math.sqrt(...)
    length_b = math.sqrt(...)
    if ... or ...:
        return 0.0
    return ...
```
**You get:** two lists of numbers, `a` and `b`, with the same length.
**You give back:** one float, the cosine similarity.

**Idea:**
1. The *dot product* multiplies the numbers in the same position and adds the results. `zip(a, b)` hands you pairs: for `[1, 2]` and `[3, 4]` you get `(1, 3)` and `(2, 4)`. So the dot product is `1*3 + 2*4 = 11`.
2. The *length* of a vector is the square root of the sum of its squares. `math.sqrt(sum(x * x for x in a))`. For `[3, 4]` it is `sqrt(9 + 16) = 5`.
3. A vector of all zeros has length 0. You cannot divide by 0, so if either length is 0, give back `0.0`.
4. Otherwise the answer is `dot / (length_a * length_b)`.

- Two vectors that point the same way give 1.0, even if one is much longer. `[1, 2, 3]` and `[2, 4, 6]` score 1.0.
- Two vectors with nothing in common, like `[1, 0]` and `[0, 1]`, give 0.0.

**Check (by hand):** `cosine_similarity([3, 4], [3, 4])` is `25 / (5 * 5) = 1.0`.

### What the asserts below test
The cell below the function checks five things:
- the same vector against itself gives `1.0`;
- `[1, 0]` against `[0, 1]` gives `0.0`;
- `[1, 2, 3]` against `[2, 4, 6]` gives `1.0` (rounded to 6 decimal places);
- opposite vectors give `-1.0`;
- a vector of zeros gives `0.0` and does not crash.

If all five hold, you see `All checks passed.`.

In [ ]:
import math


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """How much two vectors point the same way, from -1.0 to 1.0.

    You get: two lists of numbers of the same length.
    You give back: dot(a, b) / (length of a * length of b), or 0.0 if either length is 0.
    """
    # TODO:
    # 1. dot = sum(x * y for x, y in zip(a, b))
    # 2. length_a = math.sqrt(sum(x * x for x in a)), and the same for length_b.
    # 3. If either length is 0, give back 0.0.
    # 4. Otherwise give back dot / (length_a * length_b).
    raise NotImplementedError("cosine_similarity() is not implemented yet - replace the TODO with your code")


assert round(cosine_similarity([3, 4], [3, 4]), 6) == 1.0, "cosine_similarity([3, 4], [3, 4]) should be 1.0, got " + repr(cosine_similarity([3, 4], [3, 4]))
assert round(cosine_similarity([1, 0], [0, 1]), 6) == 0.0, "cosine_similarity([1, 0], [0, 1]) should be 0.0, got " + repr(cosine_similarity([1, 0], [0, 1]))
assert round(cosine_similarity([1, 2, 3], [2, 4, 6]), 6) == 1.0, "cosine_similarity([1, 2, 3], [2, 4, 6]) should be 1.0, got " + repr(cosine_similarity([1, 2, 3], [2, 4, 6]))
assert round(cosine_similarity([1, 0], [-1, 0]), 6) == -1.0, "cosine_similarity([1, 0], [-1, 0]) should be -1.0, got " + repr(cosine_similarity([1, 0], [-1, 0]))
assert cosine_similarity([0, 0], [1, 2]) == 0.0, "cosine_similarity([0, 0], [1, 2]) should be 0.0, got " + repr(cosine_similarity([0, 0], [1, 2]))

print("All checks passed.")


## Solution

In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(x * x for x in b))
    if length_a == 0 or length_b == 0:
        return 0.0
    return dot / (length_a * length_b)


print(cosine_similarity([3, 4], [3, 4]))


## Real-world project (offline)

The code cell below loads six facts about Ada and a tiny table of word vectors. Each word has 5 numbers for how much it means food, tech, sport, travel or school. Your project turns whole sentences into vectors, then finds which fact is closest in meaning to each question, using your `cosine_similarity`. Nothing here calls an LLM. This is exactly what an embedding search does, only smaller.

In [ ]:
"""Load the twin's facts and the toy word vectors (embedded so this notebook needs no repo checkout)."""
# lab-data-loader
# Each word gets 5 numbers: how much it means food, tech, sport, travel, school.
WORD_VECTORS = {
    # food
    "food": [1.0, 0.0, 0.0, 0.0, 0.0],
    "cook": [0.9, 0.0, 0.0, 0.0, 0.0],
    "cooks": [0.9, 0.0, 0.0, 0.0, 0.0],
    "bakes": [0.9, 0.0, 0.0, 0.0, 0.0],
    "bread": [1.0, 0.0, 0.0, 0.0, 0.0],
    "soup": [1.0, 0.0, 0.0, 0.0, 0.0],
    "spicy": [0.8, 0.0, 0.0, 0.1, 0.0],
    "eat": [0.9, 0.0, 0.1, 0.0, 0.0],
    # tech
    "python": [0.0, 1.0, 0.0, 0.0, 0.0],
    "code": [0.0, 0.9, 0.0, 0.0, 0.0],
    "builds": [0.0, 0.7, 0.0, 0.0, 0.0],
    "web": [0.0, 0.9, 0.0, 0.1, 0.0],
    "apps": [0.0, 1.0, 0.0, 0.0, 0.0],
    "programming": [0.0, 1.0, 0.0, 0.0, 0.0],
    "language": [0.0, 0.6, 0.0, 0.1, 0.0],
    "data": [0.0, 0.7, 0.0, 0.0, 0.3],
    "machine": [0.0, 0.7, 0.0, 0.0, 0.3],
    "learning": [0.0, 0.4, 0.0, 0.0, 0.6],
    # sport
    "runs": [0.0, 0.0, 1.0, 0.0, 0.0],
    "run": [0.0, 0.0, 1.0, 0.0, 0.0],
    "kilometers": [0.0, 0.0, 0.8, 0.2, 0.0],
    "park": [0.0, 0.0, 0.6, 0.2, 0.0],
    "team": [0.0, 0.0, 0.9, 0.0, 0.0],
    "match": [0.0, 0.0, 1.0, 0.0, 0.0],
    "watches": [0.0, 0.1, 0.4, 0.0, 0.0],
    "sport": [0.0, 0.0, 1.0, 0.0, 0.0],
    "football": [0.0, 0.0, 1.0, 0.0, 0.0],
    "exercise": [0.0, 0.0, 1.0, 0.0, 0.0],
    # travel
    "travelled": [0.0, 0.0, 0.0, 1.0, 0.0],
    "plane": [0.0, 0.0, 0.0, 1.0, 0.0],
    "accra": [0.0, 0.0, 0.0, 0.9, 0.0],
    "lisbon": [0.0, 0.0, 0.0, 0.9, 0.0],
    "summer": [0.0, 0.0, 0.0, 0.5, 0.0],
    "countries": [0.0, 0.0, 0.0, 1.0, 0.0],
    "visit": [0.0, 0.0, 0.0, 1.0, 0.0],
    "trip": [0.0, 0.0, 0.0, 1.0, 0.0],
    # school
    "teaches": [0.0, 0.0, 0.0, 0.0, 1.0],
    "teach": [0.0, 0.0, 0.0, 0.0, 1.0],
    "classes": [0.0, 0.0, 0.0, 0.0, 1.0],
}

TWIN_FACTS = [
    "Ada writes Python code and builds web apps every day.",
    "She bakes bread and cooks spicy soup on weekends.",
    "On Saturdays she runs five kilometers in the park.",
    "Last summer she travelled to Accra and Lisbon by plane.",
    "She teaches evening classes about data and machine learning.",
    "Her favorite team is Hearts of Oak and she watches every match.",
]

print(f"Loaded {len(TWIN_FACTS)} facts and {len(WORD_VECTORS)} word vectors.")


In [ ]:
def embed(text):
    words = text.lower().replace(".", "").replace("?", "").split()
    known = [WORD_VECTORS[w] for w in words if w in WORD_VECTORS]
    return [sum(column) / len(known) for column in zip(*known)]


fact_vectors = [embed(fact) for fact in TWIN_FACTS]
questions = [
    "What food does she cook?",
    "Which countries did she visit?",
    "What programming language does she use?",
]

for question in questions:
    query = embed(question)
    scores = [cosine_similarity(query, vector) for vector in fact_vectors]
    best = scores.index(max(scores))
    print(f"Q: {question}")
    print(f"A: {TWIN_FACTS[best]} ({scores[best]:.3f})")
